# 7.17 為什麼先大量讀文章，再練習當助理？


一個人讀過許多食譜，不代表接到「請列出三種不含花生的點心」時，就會直接列三項。他可能接著介紹食材，也可能沿用文章的開場白。模型也有類似的差別：文章裡合理的下一句，未必是這位使用者需要的回答。[7.11](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html)已讓我們看見，可以沿用同一個模型，換成問答示範繼續訓練。現在要問的是：為什麼常見的大模型配方把這兩件事分開做？

先看兩份教學材料。第一份是一般文章，例如「花生可用來製作糖果」。只要取得、清理文字，文章本身就提供「前文後面實際寫了什麼」這個答案；不必另外請人替每個位置寫標籤。模型反覆練習這種預測，可以逐漸學到語言、知識關聯與各種文字中的做法。這個大量學習的階段叫**預訓練**，英文pre-training。資料是否近似重複、同一家族是否混入訓練與測試兩邊，要按[5.12](https://birdhackor.github.io/tiny-perceptron-vlm/5.12.html)檢查；這項檢查不會替我們判斷文章內容是否真實。自動取得答案不代表資料免費或正確。

第二份是「使用者要求→理想回答」，還要確認有沒有避開花生、是否真的只有三項。這種材料更貼近助理的工作，卻需要設計題目、撰寫或篩選回答、核對要求。大量閱讀得到的同一組權重，可以接著用這些材料更新；這個後續階段叫**後訓練**，英文post-training。[7.1](https://birdhackor.github.io/tiny-perceptron-vlm/7.1.html)的SFT是其中一種方法。它讓訓練訊號更集中在使用者希望模型完成的任務上。

下面只做資料成本的透明算例。假設已整理好的文章每筆需要1個處理單位，經人工核對的問答每筆需要20個單位；數字是為了算清楚取捨，沒有宣稱是實際金錢或產業平均。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
article_count, checked_chat_count = 1000, 50
article_cost, checked_chat_cost = 1, 20
print("文章整理成本", article_count * article_cost)
print("問答整理成本", checked_chat_count * checked_chat_cost)
print("若1000筆都逐筆核對問答", article_count * checked_chat_cost)

輸入是筆數與假設的每筆成本。三行分別是1000、1000、20000：少量需要仔細核對的問答，已經能花掉與大量文章相同的整理預算。這段沒有訓練模型，也沒計算訓練運算費用；它說明為何資料取得方式會影響配方，不能證明「先預訓練一定比較省」或「50筆問答一定夠」。兩階段的實際成效，仍要用固定預算與留出題檢查。

另一個理由是重用。已經學到不少通用模式的底座，可以保存成一份checkpoint，也就是可重新載入的權重檔，再分別教成客服、程式助理或圖文模型。下面的圖從大量文章走到一份共用底座，之後分叉成不同用途；每條分支都會有自己的資料、更新與評測。箭頭表示繼續訓練，不是把「客服」標籤貼上去就得到能力。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/posttrain_stages.svg")))

這個分工可以從[InstructGPT原論文的引言與3.1節](https://arxiv.org/pdf/2203.02155v1)看見：預測一般文字與有用地遵循指令，目標並不相同；作者從已預訓練模型接上示範與回饋訓練。這是常見配方的具體案例，不代表所有模型都必須跑一樣的階段。後訓練也可能提高推理等任務能力，不能把它縮成只學禮貌與格式，下一節會繼續說明。

本教材的[7.12實測](https://birdhackor.github.io/tiny-perceptron-vlm/7.12.html)讓這條路變得可操作：同一個小世界文字模型，最後十題從對話微調前1題答對，變成微調後7題答對。但它的文字階段只讀同一批45題的材料，沒有學到廣泛知識；與直接SFT的5／10相比，它還多了250次文字更新。因此這份實驗驗證了接續流程與這次成績，沒有隔離產業採用兩階段的因果理由。[19.4](https://birdhackor.github.io/tiny-perceptron-vlm/19.4.html)會把保存與接續用到整合模型，而不是要求你先相信兩階段必勝。

練習只把`checked_chat_cost`改成5，先預測問答整理成本變250、全部逐筆核對變5000，再執行核對。接著寫一句這個算例還缺什麼才能選配方：至少缺實際資料品質、訓練成本與留出表現。這些不是用成本比例就能代填的數字。
